# Molecular Energy (GMPS/MPO) — Quickstart

This notebook shows the fastest path to computing molecular ground-state energies
with the Qumulator **GMPS/MPO engine** via the SDK.

The `client.molecular.energy()` method:
- Accepts one-body (`h1e`) and two-body (`h2e`) integrals in the active-space basis
- Returns the GMPS/MPO ground-state energy, qubit count, and ZZ correlators
- Works with any molecular geometry obtainable from PySCF

**Documentation**: https://sdk.qumulator.com/docs/molecular-gmps

---
Install:
```bash
pip install qumulator-sdk pyscf
```

In [ ]:
from qumulator import QumulatorClient
from pyscf import gto, scf, mcscf, ao2mo
import numpy as np

client = QumulatorClient()  # reads QUMULATOR_API_URL + QUMULATOR_API_KEY from env

## Step 1: Prepare integrals with PySCF

In [ ]:
mol = gto.M(atom="H 0 0 0; H 0 0 0.74", basis="sto-3g", spin=0, verbose=0)
mf  = scf.RHF(mol); mf.verbose = 0; mf.run()

mc  = mcscf.CASSCF(mf, ncas=2, nelecas=2); mc.verbose = 0; mc.run()

h1e, e_core = mc.get_h1eff()
h2e  = ao2mo.restore(1, mc.get_h2eff(), mc.ncas)
e_nuc = mc.energy_nuc() + e_core

print(f"h1e shape: {h1e.shape}   h2e shape: {h2e.shape}")

## Step 2: Call the GMPS/MPO engine

In [ ]:
result = client.molecular.energy(
    h1e=h1e.tolist(),
    h2e=h2e.tolist(),
    n_elec=[1, 1],          # (n_alpha, n_beta)
    e_nuc=float(e_nuc),
)

print(f"E(GMPS)       = {result.energy:.10f} Ha")
print(f"n_qubits      = {result.n_qubits}")
print(f"n_components  = {result.n_components}")

## Step 3: Compare with PySCF FCI

In [ ]:
from pyscf import fci
fcisolver = fci.FCI(mol, mf.mo_coeff); fcisolver.verbose = 0
E_fci, _ = fcisolver.kernel()

print(f"E(FCI)        = {E_fci:.10f} Ha")
print(f"|GMPS − FCI|  = {abs(result.energy - E_fci):.2e} Ha")

## Step 4: With Givens rotation circuit

Passing an orbital-rotation circuit improves the trial state used by the GMPS engine,
enabling accurate correlation recovery for strongly correlated molecules.

In [ ]:
# Minimal Givens circuit: single rotation mixing the two active orbitals
# In practice, build this from the CASSCF MO rotation matrix
circuit = [
    {"qi": 0, "qj": 1, "theta": 0.3},
]

result_c = client.molecular.energy(
    h1e=h1e.tolist(),
    h2e=h2e.tolist(),
    n_elec=[1, 1],
    e_nuc=float(e_nuc),
    circuit=circuit,
)

print(f"E(GMPS+circ)  = {result_c.energy:.10f} Ha")
print(f"n_components  = {result_c.n_components}  (may increase with circuit depth)")

## Result fields

| Field | Type | Description |
|---|---|---|
| `energy` | float | Ground-state energy in Hartrees |
| `n_qubits` | int | Number of spin-orbitals (qubits) = 2 × n_orb |
| `n_orb` | int | Number of active orbitals |
| `n_components` | int | MPS bond dimension (entanglement measure) |
| `zz_correlators` | list[list[float]] \| None | ⟨ZᵢZⱼ⟩ correlator matrix |

See full docs at [sdk.qumulator.com/docs/molecular-gmps](https://sdk.qumulator.com/docs/molecular-gmps).